# 02 — Silver: typed, cleaned, deduplicated

Where judgement gets applied. Four things happen:

1. Strings become real types; the literal `'null'` becomes a true NULL
2. Sentinel dates are resolved
3. Duplicate client rows (contract renewals) are collapsed
4. **`is_bulk_load` is derived** — the single transformation that changes
   every conclusion downstream

Redundant string date columns (`created_at_str`, `closed_at_str`) are dropped
here rather than at ingestion, so Bronze stays a faithful copy of the source.

In [0]:
from pyspark.sql import functions as F, Window
from transforms import denull, add_is_bulk_load, flag_placeholder_competitor, build_client_scd2

spark.sql("CREATE SCHEMA IF NOT EXISTS silver.bid")

bronze_bids = spark.table("bronze.bid.bids")
bronze_clients = spark.table("bronze.bid.clients")

## Null handling

The export writes the four-character string `'null'`, and `'-'` for an
absent closure date. Neither is a NULL to Spark, so both would silently
survive every downstream filter.

In [0]:
bids = denull(bronze_bids)
clients = denull(bronze_clients)

## Typing

`outcome` is deliberately left nullable: NULL means the bid is still open,
which is a distinct state from lost and must not collapse into `0`. Roughly
28% of the table sits in that state.

`to_timestamp` is given an explicit format rather than left to infer. The
default parser is version-dependent and returns NULL on a miss instead of
raising — exactly the kind of silent data loss this pipeline is built to
avoid. The assert right after fails loudly if that ever happens.

In [0]:
TS_FORMAT = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

bids_typed = (
    bids
    .withColumn("bid_id", F.col("bid_id").cast("long"))
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("created_at", F.to_timestamp("created_at", TS_FORMAT))
    .withColumn("bid_date", F.to_timestamp("bid_date", TS_FORMAT))
    .withColumn("closed_at", F.to_timestamp("closed_at", TS_FORMAT))
    .withColumn("outcome", F.col("outcome").cast("int"))          # NULL = open
    .withColumn("is_confirmed_date", F.col("is_confirmed_date").cast("int"))
    .withColumn("contract_value_brl", F.col("contract_value_brl").cast("double"))
    .drop("created_at_str", "closed_at_str")
)

# Guard: a created_at that was non-NULL before the cast but NULL after it
# means the format string stopped matching the source — fail loudly, not
# silently.
bad_created = (
    bids.filter(F.col("created_at").isNotNull())
    .join(bids_typed.filter(F.col("created_at").isNull()), "bid_id", "inner")
    .count()
)
assert bad_created == 0, (
    f"{bad_created} created_at values failed to parse with format {TS_FORMAT} "
    "— check for a source date format change."
)

## Deriving `is_bulk_load`

A large share of rows were mass-imported during a system migration rather
than entered as bids happened. They share an identical `created_at` down to
the second, and they behave nothing like organic bids — a far lower win rate,
concentrated in specific portfolios.

Left unflagged, they poison every segmented metric: the executive who owned
the migrated portfolio looks like the worst performer in the company purely
because of how their records were loaded.

The threshold of 10 is a judgement call. Two bids registered in the same
second is plausible; ten is not.

In [0]:
BULK_THRESHOLD = 10

bids_flagged = add_is_bulk_load(bids_typed, threshold=BULK_THRESHOLD)

## Loss reason coverage

`competitor_name` carries a default value written whenever nobody completed
the post-mortem. Flagging it explicitly stops it being counted as a real
competitor in any downstream aggregate — which would otherwise produce the
false headline that one competitor takes the overwhelming majority of losses.

In [0]:
PLACEHOLDER_COMPETITOR = "Competitor 1"

bids_clean = (
    flag_placeholder_competitor(bids_flagged, placeholder=PLACEHOLDER_COMPETITOR)
    .withColumn(
        "bid_status",
        F.when(F.col("outcome") == 1, "won")
         .when(F.col("outcome") == 0, "lost")
         .otherwise("open"),
    )
)

bids_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.bids_clean")

## Clients: sentinel dates and SCD Type 2

`2999-12-31` marks an open-ended contract; keeping it as a date would put a
977-year contract into any duration calculation. It becomes NULL alongside an
explicit `is_open_ended` flag.

Renewals are a second row for the same `client_id`, with a `start_date`
genuinely later than the first contract's `end_date` — a real second
validity window, not a same-day duplicate. Collapsing to "most recent row"
(the old approach) threw the earlier period away entirely. This keeps every
version and gives each one an explicit `valid_from` / `valid_to`:

- `valid_from` = that version's `start_date`
- `valid_to` = the *next* version's `start_date` for that client, or NULL if
  there isn't one — NULL means **currently valid**, not "unknown"
- `is_current` = `valid_to IS NULL`
- `client_sk` = a surrogate key identifying a (client, version) pair —
  `client_id` stays the business key for joins that only care about *who*,
  not *which version*

Gold decides whether it wants the current version or the version that was
valid when a given bid was placed. Silver's job stops at making both
possible.

In [0]:
SENTINEL = "2999-12-31"

clients_clean = build_client_scd2(clients, sentinel=SENTINEL)

spark.sql("DROP TABLE IF EXISTS silver.bid.clients_clean")
clients_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.bid.clients_clean")

clients_clean = spark.table("silver.bid.clients_clean")

# Every client_id should have exactly one current version — guaranteed by
# construction (the last row in each partition always has valid_to = NULL),
# checked explicitly anyway because a silent violation here would silently
# break every point-in-time join downstream.
bad_versioning = (
    clients_clean.groupBy("client_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("n_current"))
    .filter("n_current != 1")
    .count()
)
assert bad_versioning == 0, f"{bad_versioning} client_ids don't have exactly one current version"

In [0]:
%pip install openpyxl

import pandas as pd
pdf = pd.read_excel("/Volumes/raw/bid/bids/clients.xlsx", sheet_name="Clients")
dupes = pdf[pdf["contract_name"].str.contains("renewal", na=False)]
sample = dupes.iloc[0]
original = pdf[(pdf["client_id"] == sample["client_id"]) & (~pdf["contract_name"].str.contains("renewal"))]
print(sample[["client_id","start_date","end_date"]])
print(original[["client_id","start_date","end_date"]])

## Referential integrity

A bid pointing at a client that does not exist would be dropped silently by
an inner join later. Checking here means it surfaces as a number, not as a
quietly shrinking row count.

The point-in-time check below is expected to find a non-zero, non-trivial
count — roughly 6% of bids. Two real causes, not a bug in this join:
some clients have an unknown `start_date` (the `'-'` placeholder) and can't
be placed on a timeline at all, and some bids were created *before* the
client's earliest known contract start — a genuine inconsistency in the
source data that a simple `client_id` join was silently papering over by
attaching whichever version happened to be "latest," date be damned. The
left join in Gold keeps these bids with NULL client attributes rather than
dropping them or guessing.

In [0]:
# Referential integrity: does the client exist at all? (Any version —
# temporal validity is a separate concern, checked next.)
orphans = (
    bids_clean.join(clients_clean.select("client_id").distinct(), "client_id", "left_anti").count()
)
print(f"orphan bids: {orphans}")

# Point-in-time coverage: does every bid's created_at actually fall inside
# some version's [valid_from, valid_to) window for its client? A gap here
# would mean Gold's point-in-time join drops that bid's client attributes
# silently.
coverage_check = (
    bids_clean.alias("b")
    .join(
        clients_clean.alias("c"),
        (F.col("b.client_id") == F.col("c.client_id"))
        & (F.col("b.created_at") >= F.col("c.valid_from").cast("timestamp"))
        & (F.col("c.valid_to").isNull() | (F.col("b.created_at") < F.col("c.valid_to").cast("timestamp"))),
        "left_anti",
    )
)
uncovered = coverage_check.count()
print(f"bids with no matching client version at their created_at: {uncovered}")